# Configuración para Google Colab
Ejecutar esta sección antes que el resto del notebook.

In [ ]:
# Montar Google Drive para persistir datos entre sesiones
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# Posicionarse en la carpeta de Drive donde están los experimentos
CARPETA_EXP = '/content/drive/MyDrive/IA_Experimentos'
os.makedirs(CARPETA_EXP, exist_ok=True)
os.chdir(CARPETA_EXP)

In [ ]:
# Clonar el repo si no existe (solo la primera vez)
if not os.path.exists('sistema-control-asistencia-facial'):
    !git clone https://github.com/MiguelTacoZavala/sistema-control-asistencia-facial.git

os.chdir('sistema-control-asistencia-facial')
!git pull  # traer cambios recientes si los hay

In [ ]:
# Instalar dependencias del proyecto
!pip install -r requirements.txt

---
# Experimento 1: Detección Base — YOLOv8
**Objetivo:** Establecer el punto de referencia del rendimiento del detector YOLO
sin reconocimiento ni tracking. Se mide FPS, confianza y cantidad de rostros
detectados por frame sobre al menos 2 videos de prueba.

In [ ]:
import sys
import time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import pandas as pd

# En Colab ya hicimos %cd al repo raíz. En local, el notebook
# está en experiments/ así que la raíz es un nivel arriba.
IN_COLAB = 'google.colab' in sys.modules
RAIZ = Path.cwd() if IN_COLAB else Path.cwd().parent
sys.path.insert(0, str(RAIZ))

from src.detector import FaceDetector

print(f"Raíz del proyecto: {RAIZ}")
print("Imports completados.")

In [ ]:
# Configurar detector con umbral por defecto
RUTA_MODELO = RAIZ / "models" / "yolov8n-face.pt"
detector = FaceDetector(str(RUTA_MODELO), conf_threshold=0.5)

print(f"Modelo cargado desde: {RUTA_MODELO}")
print(f"Umbral de confianza: {detector.conf_threshold}")

In [ ]:
# Listar videos de prueba
RUTA_VIDEOS = RAIZ / "dataset" / "test_videos"
videos = sorted(RUTA_VIDEOS.glob("*.mp4"))

if not videos:
    raise FileNotFoundError(f"No se encontraron videos .mp4 en {RUTA_VIDEOS}")

print(f"Videos encontrados ({len(videos)}):")
for v in videos:
    print(f"  - {v.name}")

In [ ]:
%%time

resultados_por_video = []

for ruta_video in videos:
    nombre_video = ruta_video.name
    print(f"\nProcesando: {nombre_video}")

    cap = cv2.VideoCapture(str(ruta_video))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps_video = cap.get(cv2.CAP_PROP_FPS)
    print(f"  Frames totales: {total_frames} | FPS del video: {fps_video:.2f}")

    tiempos_inferencia = []
    confianzas = []
    rostros_por_frame = []

    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        t_inicio = time.perf_counter()
        detecciones = detector.detect(frame)
        t_fin = time.perf_counter()

        tiempo_ms = (t_fin - t_inicio) * 1000
        tiempos_inferencia.append(tiempo_ms)

        rostros_por_frame.append(len(detecciones))

        for det in detecciones:
            confianzas.append(det[4])

        frame_idx += 1
        if frame_idx % 100 == 0:
            print(f"    {frame_idx}/{total_frames} frames procesados")

    cap.release()

    # Calcular métricas del video
    n = len(tiempos_inferencia)
    if n == 0:
        print(f"  [SKIP] {nombre_video} — sin frames")
        continue

    fps_list = [1000 / t if t > 0 else 0 for t in tiempos_inferencia]
    fps_promedio = sum(fps_list) / n
    fps_min = min(fps_list)
    fps_max = max(fps_list)

    conf_promedio = sum(confianzas) / len(confianzas) if confianzas else 0.0
    rostros_promedio = sum(rostros_por_frame) / n

    resultados_por_video.append({
        "video": nombre_video,
        "frames": n,
        "fps_promedio": round(fps_promedio, 2),
        "fps_min": round(fps_min, 2),
        "fps_max": round(fps_max, 2),
        "confianza_promedio": round(conf_promedio, 3),
        "rostros_por_frame": round(rostros_promedio, 2),
        "inferencia_promedio_ms": round(sum(tiempos_inferencia) / n, 2),
    })

    print(f"  Procesado: {n} frames en total")
    print(f"  FPS promedio: {fps_promedio:.2f} | Inferencia: {sum(tiempos_inferencia)/n:.2f} ms")

In [ ]:
# Tabla resumen con pandas
df_resultados = pd.DataFrame(resultados_por_video)
display(df_resultados)

In [ ]:
# Gráfico de FPS a lo largo del tiempo
fig, ax = plt.subplots(figsize=(12, 5))

for ruta_video in videos:
    nombre = ruta_video.name
    cap = cv2.VideoCapture(str(ruta_video))

    fps_frame = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        t0 = time.perf_counter()
        _ = detector.detect(frame)
        t1 = time.perf_counter()
        t_ms = (t1 - t0) * 1000
        fps_frame.append(1000 / t_ms if t_ms > 0 else 0)

    cap.release()
    ax.plot(fps_frame, label=nombre, linewidth=0.7)

ax.axhline(y=15, color='r', linestyle='--', alpha=0.5, label='Umbral tiempo real (15 FPS)')
ax.set_xlabel("Frame")
ax.set_ylabel("FPS")
ax.set_title("FPS de detección a lo largo del tiempo")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RAIZ / "experiments" / "graphs" / "fps_tiempo_real.png", dpi=150)
plt.show()

In [ ]:
# Exportar resultados a CSV
ruta_csv = RAIZ / "experiments" / "resultados_exp1.csv"
df_resultados.to_csv(ruta_csv, index=False)
print(f"Resultados guardados en: {ruta_csv}")

## Interpretación de resultados

**[Completar después de ejecutar]**

### ¿Qué tan rápido es el detector?

El FPS promedio obtenido fue de **[X] fps**, con un mínimo de **[Y] fps** y un máximo de **[Z] fps**.
El tiempo de inferencia por frame fue de aproximadamente **[W] ms**. Esto significa que el detector
por sí solo procesa cada frame en **[W] ms**, lo que permite alcanzar **[A] fps** en promedio.

### ¿Es suficiente para tiempo real?

Generalmente se considera que un sistema funciona en "tiempo real" para video cuando supera
los **15 FPS** (experiencia visual fluida) o al menos **10 FPS** (usable con leve latencia).
Nuestro detector alcanzó **[X] FPS** en promedio, lo que significa que **sí / no** es suficiente
para tiempo real. Es importante notar que estos valores son **sin** reconocimiento ni tracking;
al agregar esos módulos los FPS disminuirán, por lo que tener un margen amplio es favorable.

### Variación entre videos

Los resultados variaron entre los dos videos: **[video1]** presentó **[X] fps** mientras que
**[video2]** alcanzó **[Y] fps**. Esta diferencia puede deberse a la resolución del video,
la cantidad de rostros por frame, la iluminación o la duración del video. El video con más
rostros por frame tiende a tener una inferencia ligeramente más lenta, aunque YOLOv8n está
optimizado para detectar múltiples objetos en una sola pasada.

*Esta interpretación debe ajustarse con los valores reales obtenidos tras la ejecución.*